# Validate spectra (buoy vs reconstruction)

Compare **observed buoy spectra** to reconstructed metamodel spectra:

- **Buoy** — processed NetCDF (`Efth` on native freq/dir grid)
- **Bulk reconstruct** — `mm.reconstruct(..., spectral=False)` + `save_spectra`
- **Partition reconstruct** — `mm.reconstruct(..., spectral=True)` (WHACS partitions 0–3 summed)

Every figure below is the same **3 columns, 1 row** layout — buoy, bulk, spectral,
always in that order, left to right — for one "moment" (one random timestep, one
monthly mean, one seasonal mean). However many moments get plotted, each one is its
own side-by-side comparison, so the same thing is always directly comparable instead
of scattered across separate per-source figures (`bluemath_tk.core.plotting.spectra`;
wavespectra's own `.spec.plot()` can't target a shared subplot axis, so each panel is
rendered independently and composed as images — see that module's docstring).

**Prerequisites:** spectra caches under `tmp/reconstruct/` for the same `period`,
`sites`, and `mda_n` (`save_spectra: true` in the case JSON).

In [ ]:
import numpy as np
import xarray as xr
from IPython.display import display
from bluemath_tk.core.plotting.spectra import plot_spectrum_comparison
from utils.config import resolve
from utils import metamodel as mm
from utils import onshore_sites as obs

## Configuration

In [ ]:
period = "2020-07"
offshore_name = "WHACS"
mda_n = 450
mm_set = "buoys"  # metamodel set (metamodel.sets) fitted on these sites
wl = 0.0
goals = None  # all goals

# instrument_id values from notebook 03 — add more to compare several buoys
# side by side (each gets its own row of figures below).
buoy_instruments = ["TODO_Buoy"]
sites = [obs.snapwave_site_id(iid) for iid in buoy_instruments]

buoy_raw_files = {
    obs.snapwave_site_id("TODO_Buoy"): "inputs/Observations/processed/TODO_buoy.nc",
}

n_random = 12
random_seed = 42

recon_kw = dict(
    n=mda_n,
    metamodel=mm_set,
    offshore_name=offshore_name,
    goals=goals,
    wl=wl,
    sites=sites,
)

print("Observations JSON:", obs.geojson_path())
print(f"period   : {period}")
print(f"sites    : {sites}")

## 1 · Load spectra

In [ ]:
buoy_efth: dict[str, xr.DataArray] = {}
for s in sites:
    raw = xr.open_dataset(resolve(buoy_raw_files[s])).sel(time=period)
    if "Efth" in raw or "efth" in raw:
        spec_var = "Efth" if "Efth" in raw else "efth"
    else:
        # Some processed files carry only a peak-normalized shape (no
        # absolute-scale efth) — fall back to it. Every panel below gets
        # renormalized to peak=1 for display anyway, so this is still a
        # fair shape comparison.
        spec_var = "efth_norm"
        print(f"{s}: no absolute 'efth' — using 'efth_norm' (normalized shape only)")
    buoy_efth[s] = raw[spec_var]

# Reconstruct once for all buoys together so this cache is reusable across
# sites. Cheap if already cached from a previous run of this cell;
# otherwise budget a while the first time (one mm.reconstruct() per mode,
# spanning `period`). Both modes need save_spectra=True — "bulk"
# (spectral=False) still writes its own efth companion (single WHACS
# partition -1), same as the partitioned mode.
mm.reconstruct(period, spectral=False, save_spectra=True, **recon_kw)
mm.reconstruct(period, spectral=True, save_spectra=True, **recon_kw)

bulk_spec = mm.load_reconstruction_spectra(period, spectral=False, **recon_kw)
part_spec = mm.load_reconstruction_spectra(period, spectral=True, **recon_kw)

bulk_efth = {s: bulk_spec["efth"].sel(sites=s) for s in sites}
part_efth = {s: part_spec["efth"].sel(sites=s) for s in sites}

for s in sites:
    print(
        f"{s}: buoy={dict(buoy_efth[s].sizes)}  "
        f"bulk={dict(bulk_efth[s].sizes)}  part={dict(part_efth[s].sizes)}"
    )

## 2 · Helpers

In [ ]:
def pick_random_time_index(n: int, n_time: int, seed: int) -> np.ndarray:
    """Sorted integer indices along ``time``."""
    rng = np.random.default_rng(seed)
    n_pick = min(n, n_time)
    return np.sort(rng.choice(n_time, size=n_pick, replace=False))


def monthly_mean_spectra(da: xr.DataArray) -> dict[str, xr.DataArray]:
    """One time-averaged spectrum per calendar month present, keyed ``month=N``.

    Groups by calendar month across every year in ``da`` (a climatology, not
    one panel per year-month) so this stays at 12 panels regardless of how
    many years ``period`` spans.
    """
    months = sorted({int(m) for m in da["time"].dt.month.values})
    return {
        f"month={month}": da.sel(time=da.time.dt.month == month).mean(
            "time", skipna=True
        )
        for month in months
    }


def seasonal_mean_spectra(da: xr.DataArray) -> dict[str, xr.DataArray]:
    """One time-averaged spectrum per season present, keyed ``season=DJF`` etc.

    Groups by season code across every year in ``da``, so this stays at (up
    to) 4 panels regardless of how many years ``period`` spans.
    """
    seasons = sorted({str(s) for s in da["time"].dt.season.values})
    return {
        f"season={season}": da.sel(time=da.time.dt.season == season).mean(
            "time", skipna=True
        )
        for season in seasons
    }


PANEL_LABELS = (
    "Buoy observed",
    "Reconstruction bulk (partition -1)",
    "Reconstruction spectral (partitions 0-3)",
)
# Every panel is plotted with wavespectra's default normalised=True (each
# spectrum independently scaled 0-1 by its own peak) — the 3 sources' raw
# attrs can disagree on what to call that, so pin one consistent label
# rather than showing 3 differently-worded colorbars for the same 0-1
# scale. This also makes the 'efth_norm' fallback (see Configuration) a
# fair comparison — every panel ends up peak-normalized for display anyway.
CBAR_KWARGS = {"label": "Normalised energy density", "shrink": 0.8, "pad": 0.08}


def show_triplet(
    buoy_da: xr.DataArray | None,
    bulk_da: xr.DataArray | None,
    part_da: xr.DataArray | None,
    title: str,
) -> None:
    """Buoy / bulk / spectral, always 3 columns in that order, for one moment.

    Any source missing for this particular moment (e.g. a month the buoy
    has no data for) is skipped rather than plotted blank, but the other
    panels still show — and still say which one they are.
    """
    specs = [
        (label, da)
        for label, da in zip(PANEL_LABELS, (buoy_da, bulk_da, part_da), strict=True)
        if da is not None
    ]
    if not specs:
        print(f"{title}: nothing to plot (missing on all 3 sources)")
        return
    missing = [
        label
        for label, da in zip(PANEL_LABELS, (buoy_da, bulk_da, part_da), strict=True)
        if da is None
    ]
    print(f"— {title} —" + (f"  (missing: {', '.join(missing)})" if missing else ""))
    display(plot_spectrum_comparison(specs, cbar_kwargs=CBAR_KWARGS))


# One independent random draw per site (buoy coverage/time axis differs per
# site, so each gets its own index into its own time axis).
samples: dict[str, dict[str, xr.DataArray]] = {}
for s in sites:
    idx = pick_random_time_index(n_random, buoy_efth[s].sizes["time"], random_seed)
    sample_buoy = buoy_efth[s].isel(time=idx).load()
    samples[s] = {
        "buoy": sample_buoy,
        "bulk": bulk_efth[s].sel(time=sample_buoy.time.values).load(),
        "part": part_efth[s].sel(time=sample_buoy.time.values).load(),
    }
    print(f"{s}: random sample {len(idx)} timesteps (seed={random_seed})")

## 3 · Random timesteps

Independent draws on a shared hourly time axis (buoy and reconstruction times match).
One 3-column comparison per timestep — increase `n_random` above for more, the layout
doesn't change.

In [ ]:
for s in sites:
    sb, bb, pb = samples[s]["buoy"], samples[s]["bulk"], samples[s]["part"]
    for i in range(sb.sizes["time"]):
        t = sb.time.values[i]
        show_triplet(
            sb.isel(time=i),
            bb.isel(time=i),
            pb.isel(time=i),
            f"{s} — {np.datetime_as_string(t, unit='h')}",
        )

## 4 · Monthly mean spectra

In [ ]:
for s in sites:
    buoy_monthly = monthly_mean_spectra(buoy_efth[s].load())
    bulk_monthly = monthly_mean_spectra(bulk_efth[s].load())
    part_monthly = monthly_mean_spectra(part_efth[s].load())

    # Buoy coverage is the limiting factor — walk its months, not the
    # (always fully hourly) reconstruction's, and skip a source that has
    # none for it.
    for key in buoy_monthly:
        show_triplet(
            buoy_monthly[key],
            bulk_monthly.get(key),
            part_monthly.get(key),
            f"{s} — {period} — {key}",
        )

## 5 · Seasonal mean spectra

Meteorological seasons (`DJF`, `MAM`, …). For a single calendar month there is
typically one season in the sample.

In [ ]:
for s in sites:
    buoy_seasonal = seasonal_mean_spectra(buoy_efth[s])
    bulk_seasonal = seasonal_mean_spectra(bulk_efth[s])
    part_seasonal = seasonal_mean_spectra(part_efth[s])

    for key in buoy_seasonal:
        show_triplet(
            buoy_seasonal[key],
            bulk_seasonal.get(key),
            part_seasonal.get(key),
            f"{s} — {period} — {key}",
        )